In [1]:
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
import yaml

step_font_dict = {"color": "black", "fontfamily": "Arial", "fontsize": 10}

In [2]:
with open("panel_sizes_cm.yaml", "r") as file:
    panel_sizes = yaml.safe_load(file)

fig_width_cm = panel_sizes["panel_a"]["width_cm"]
fig_height_cm = panel_sizes["panel_a"]["height_cm"]

scale_panel = 1.25
fig_width_inch = fig_width_cm / 2.54 * scale_panel
fig_height_inch = fig_height_cm / 2.54 * scale_panel

In [ ]:
import re

from IPython.display import SVG
from IPython.display import display as ipython_display
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

# ---------- SVG vector icon support ----------

_icon_defs = []


def add_icon(ax, path, xy, zoom=0.1):
    """Record an icon for post-processing. Icons are injected as true SVG
    vectors after saving, so they scale perfectly on any poster or screen."""
    _icon_defs.append({"path": path, "xy": xy, "zoom": zoom})


def inject_svg_icons(svg_path, icon_defs, ax, fig):
    """Post-process the saved matplotlib SVG to embed icon SVGs as vector elements.

    Coordinate mapping
    ------------------
    matplotlib display coords: pixels at fig.dpi (origin bottom-left)
    SVG coords:                points at 72 dpi  (origin top-left, y flipped)

    Icon size
    ---------
    Matches the visual size that OffsetImage(zoom=zoom, img_width=1024) would
    produce:  target_pt = zoom * 1024  (in SVG points)
    """
    fig_h_pt = fig.get_figheight() * 72  # figure height in SVG pt
    fig_dpi = fig.dpi

    inject_parts = []
    for d in icon_defs:
        path, xy, zoom = d["path"], d["xy"], d["zoom"]

        # Data coords  ->  display pixels  ->  SVG points (y flipped)
        dx, dy = ax.transData.transform(xy)
        svg_cx = dx / fig_dpi * 72
        svg_cy = fig_h_pt - dy / fig_dpi * 72

        # Desired icon width in SVG pts
        target_pt = zoom * 1024

        # Parse the icon SVG
        with open(path) as f:
            icon_text = f.read()

        # Extract viewBox dimensions
        vb = re.search(r"""viewBox=["']\s*0\s+0\s+([\d.]+)\s+([\d.]+)["']""", icon_text)
        icon_w = float(vb.group(1)) if vb else 100.0
        icon_h = float(vb.group(2)) if vb else 100.0
        scale = target_pt / icon_w

        # Extract inner SVG content (strip outer <svg> wrapper)
        inner = re.search(r"<svg[^>]*>(.*?)</svg>", icon_text, re.DOTALL)
        if not inner:
            continue

        # Top-left corner so the icon is centred on (svg_cx, svg_cy)
        tx = svg_cx - target_pt / 2
        ty = svg_cy - (icon_h * scale) / 2

        inject_parts.append(
            f'<g transform="translate({tx:.3f},{ty:.3f}) scale({scale:.6f})">\n'
            + inner.group(1)
            + "\n</g>"
        )

    with open(svg_path) as f:
        svg_text = f.read()
    svg_text = svg_text.replace("</svg>", "\n".join(inject_parts) + "\n</svg>")
    with open(svg_path, "w") as f:
        f.write(svg_text)


# ---------- Figure ----------

# Extended figure: panel A (y in [0,3]) + loop section (y in [-5,0])
# Total y-range = 8 units vs original 3 → scale height proportionally
fig_height_extended = fig_height_cm / 2.54 * scale_panel * (8 / 3)

_icon_defs.clear()

with mpl.rc_context(fname="../../.matplotlibrc"):
    fig, ax = plt.subplots(
        figsize=(fig_width_inch, fig_height_extended), layout="constrained"
    )
    ax.set_xlim(0, 10)
    ax.set_ylim(-5, 3)
    ax.axis("off")

    # ── Panel A (unchanged) ────────────────────────────────────────────────

    # Central rounded box
    center_box = FancyBboxPatch(
        (3.0, 0.5),
        2.9,
        2,
        boxstyle="round,pad=0.02,rounding_size=0.2",
        linewidth=1.5,
        edgecolor="black",
        facecolor="#f7a1a1",
    )
    ax.add_patch(center_box)

    # Left labels
    ax.text(
        0,
        2.2,
        "Problem\nformulation",
        va="center",
        ha="center",
        zorder=10,
        fontdict=step_font_dict,
    )
    ax.text(
        0, 0.8, "Context", va="center", ha="center", zorder=10, fontdict=step_font_dict
    )

    # Right label
    ax.text(
        9,
        1.5,
        "Simulator\ncode",
        va="center",
        ha="center",
        zorder=10,
        fontdict=step_font_dict,
    )

    # Center label
    ax.text(
        4.45,
        1,
        "Discovery\ntool",
        va="center",
        ha="center",
        zorder=10,
        fontdict=step_font_dict,
    )

    # Arrows (panel A)
    arrow_kw = dict(
        arrowstyle="->", linewidth=1.5, mutation_scale=10, capstyle="butt", zorder=9
    )
    ax.add_patch(FancyArrowPatch((2.0, 2.2), (3.0, 2.2), **arrow_kw))
    ax.add_patch(FancyArrowPatch((2.0, 0.8), (3.0, 0.8), **arrow_kw))
    ax.add_patch(FancyArrowPatch((5.9, 1.5), (6.9, 1.5), **arrow_kw))

    ax.add_line(
        Line2D(
            [3.0, 3.0 + 2.9],
            [1.55, 1.55],
            linewidth=1.5,
            color="black",
            zorder=9,
            linestyle="--",
        )
    )

    # Icons (panel A)
    add_icon(ax, "icons/file-line-icon.svg", (1.7, 2.2), zoom=0.018)
    add_icon(ax, "icons/database-line-icon.svg", (1.6, 0.9), zoom=0.018)
    add_icon(ax, "icons/female-icon.svg", (4.60, 1.9), zoom=0.016)
    add_icon(ax, "icons/flask-icon.svg", (4.25, 1.93), zoom=0.007)
    add_icon(ax, "icons/monitor-icon.svg", (5.5, 1.9), zoom=0.018)
    add_icon(ax, "icons/atom-laboratory-science-icon.svg", (5.5, 1.95), zoom=0.008)
    add_icon(ax, "icons/bacteria-infection-virus-icon.svg", (3.6, 1.9), zoom=0.016)
    add_icon(ax, "icons/web-code-icon.svg", (7.5, 1.4), zoom=0.024)

    # ── Zoom connectors ────────────────────────────────────────────────────
    # Dashed diagonal lines expanding from the bottom of the central box
    # down to the top-left / top-right corners of the loop box.
    connector_kw = dict(linewidth=1.2, color="black", linestyle="--", zorder=5)
    ax.add_line(Line2D([3.0, 0.5], [0.5, -0.3], **connector_kw))  # left
    ax.add_line(Line2D([5.9, 9.5], [0.5, -0.3], **connector_kw))  # right

    # ── Loop section ───────────────────────────────────────────────────────

    # Outer green box  (set by user: anchor=(0.5,-3.8), w=9.0, h=3.5)
    ax.add_patch(
        FancyBboxPatch(
            (0.5, -3.2),
            9.0,
            2.8,
            boxstyle="round,pad=0.05,rounding_size=0.2",
            linewidth=1.5,
            edgecolor="black",
            facecolor="#d4edda",
            zorder=1,
        )
    )

    # Inner salmon boxes  (anchor = bottom-left corner)
    box_kw = dict(
        boxstyle="round,pad=0.02,rounding_size=0.15",
        linewidth=1.5,
        edgecolor="black",
        facecolor="#f7a1a1",
        zorder=3,
    )
    # LLM: Feedback  (top centre)
    ax.add_patch(FancyBboxPatch((3.0, -1.5), 3.5, 0.9, **box_kw))
    ax.text(
        4.75,
        -1.05,
        "LLM:\nFeedback",
        va="center",
        ha="center",
        zorder=10,
        fontdict=step_font_dict,
    )

    # LLM: Coding  (bottom left)
    ax.add_patch(FancyBboxPatch((1.2, -2.8), 2.2, 0.9, **box_kw))
    ax.text(
        2.3,
        -2.35,
        "LLM:\nCoding",
        va="center",
        ha="center",
        zorder=10,
        fontdict=step_font_dict,
    )

    # Parameter optimizer  (bottom centre)
    ax.add_patch(FancyBboxPatch((3.8, -2.8), 2.8, 0.9, **box_kw))
    ax.text(
        5.2,
        -2.35,
        "Parameter\noptimizer",
        va="center",
        ha="center",
        zorder=10,
        fontdict=step_font_dict,
    )

    # Evaluator  (bottom right)
    ax.add_patch(FancyBboxPatch((7.0, -2.8), 1.8, 0.9, **box_kw))
    ax.text(
        7.9,
        -2.35,
        "Evaluator",
        va="center",
        ha="center",
        zorder=10,
        fontdict=step_font_dict,
    )

    # Arrows inside the loop
    # Horizontal flow: Coding → Param optimizer → Evaluator
    ax.add_patch(FancyArrowPatch((3.4, -2.35), (3.8, -2.35), **arrow_kw))
    ax.add_patch(FancyArrowPatch((6.6, -2.35), (7.0, -2.35), **arrow_kw))

    # Evaluator (right) → up the right side → Feedback (right)
    ax.add_line(
        Line2D(
            [8.8, 9.1, 9.1],
            [-2.35, -2.35, -1.05],
            linewidth=1.5,
            color="black",
            zorder=9,
            solid_capstyle="butt",
        )
    )
    ax.add_patch(FancyArrowPatch((9.1, -1.05), (6.5, -1.05), **arrow_kw))

    # Feedback (left) → Coding (top): right-angle path then arrowhead
    ax.add_line(
        Line2D(
            [3.0, 2.3, 2.3],
            [-1.05, -1.05, -1.85],
            linewidth=1.5,
            color="black",
            zorder=9,
            solid_capstyle="butt",
        )
    )
    ax.add_patch(FancyArrowPatch((2.3, -1.85), (2.3, -1.9), **arrow_kw))

    # Multiple input arrows into LLM:Coding from the left (SMC population)
    for y_in in [-2.15, -2.55]:
        ax.add_patch(FancyArrowPatch((0.2, y_in), (1.2, y_in), **arrow_kw))

    # ── Save & inject vector icons ─────────────────────────────────────────
    save_folder = "../panels/"
    os.makedirs(save_folder, exist_ok=True)
    svg_path = os.path.join(save_folder, "panel_a_extended.svg")

    plt.savefig(svg_path, format="svg", transparent=True, dpi=300)
    inject_svg_icons(svg_path, _icon_defs, ax, fig)

plt.close(fig)

# Display the final vector SVG inline
ipython_display(SVG(svg_path))